# Purpose:
- To query lims for data to be uploaded
- For LAMF and GCaMP pilot data
- allenvisb environment (brain_observatory_qc, allensdk)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from brain_observatory_qc.data_access import from_lims
from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache as bpc

c:\Users\jinho.kim\Anaconda3\envs\allenvisb\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\jinho.kim\Anaconda3\envs\allenvisb\lib\site-packages\numpy\.libs\libopenblas.EL2C6PLE4ZYW3ECEVIV3OXXGRN2NRFM2.gfortran-win_amd64.dll
c:\Users\jinho.kim\Anaconda3\envs\allenvisb\lib\site-packages\numpy\.libs\libopenblas.gk7gx5keq4f6uyo3p26ulgbqyhgqo7j4.gfortran-win_amd64.dll
c:\Users\jinho.kim\Anaconda3\envs\allenvisb\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
cache = bpc.from_lims()
exp_table = cache.get_ophys_experiment_table(passed_only=False)

# 721291 ("Saffron")

In [26]:
# the ones in code ocean
saffron_co_df = pd.read_csv(r"\\allen\programs\mindscope\workgroups\learning\saffron_data_asset_df.csv")

In [33]:
saffron_co_df.head()

,raw_data_date,processed_date,capsule_id,commit_id,processed_data_asset_id,raw_data_asset_id
0,2024-04-16,2024-11-06,24cad997-4633-4ab3-a17d-8b0c01cad787,ea95e8423c69e5617cacd25a1620f7b47d8828be,2b02bae7-abbb-4383-a8f7-5f6fc31b19e9,1c7691f2-5d15-4c2f-a32d-27009b5abce7
1,2024-04-17,2024-11-06,24cad997-4633-4ab3-a17d-8b0c01cad787,ea95e8423c69e5617cacd25a1620f7b47d8828be,4251a7d8-1190-4064-8ead-d52cb9a8be4a,76b4c367-dd11-46f6-bdbd-1b2c9f60ffa9
2,2024-04-19,2024-11-06,24cad997-4633-4ab3-a17d-8b0c01cad787,ea95e8423c69e5617cacd25a1620f7b47d8828be,b13e52c4-2bf1-4d1f-bac8-6804d3019b81,94f21415-497a-4dc3-bd33-291a1c67127d
3,2024-04-22,2024-11-06,24cad997-4633-4ab3-a17d-8b0c01cad787,ea95e8423c69e5617cacd25a1620f7b47d8828be,6fa3d737-f6f4-4317-a20b-9ecf6252270f,e2a394f1-acaf-45e8-8ffb-e2cb0a35df1d
4,2024-04-23,2024-11-06,24cad997-4633-4ab3-a17d-8b0c01cad787,ea95e8423c69e5617cacd25a1620f7b47d8828be,453046c2-ce92-4cd6-9282-6be86c25d1cf,25331ba5-3531-4996-bbff-5a12e2fab271


In [54]:
save_dir = Path(r'\\allen\programs\mindscope\workgroups\learning')
saffron_df = exp_table.query('mouse_id == "721291" and experiment_workflow_state != "created"').reset_index()
# there is one duplicate
saffron_df = saffron_df.drop_duplicates(subset='ophys_experiment_id')
saffron_session_df = saffron_df[['ophys_session_id']]

saffron_lims_df = saffron_df[['ophys_session_id', 'date_of_acquisition', 'session_type']].drop_duplicates().copy()
saffron_lims_df['date_of_acquisition'] = saffron_lims_df['date_of_acquisition'].apply(lambda x: str(x).split(' ')[0])
saffron_lims_df.sort_values(by='date_of_acquisition', inplace=True)
saffron_lims_df.head()

,ophys_session_id,date_of_acquisition,session_type
8,1345647212,2024-04-16,TRAINING_0_gratings_autorewards_15min
16,1345978020,2024-04-17,TRAINING_1_gratings
72,1346444914,2024-04-19,TRAINING_1_gratings
24,1347023256,2024-04-22,TRAINING_1_gratings
40,1347238498,2024-04-23,TRAINING_2_gratings_flashed


In [57]:
# saffron_to_upload_df = saffron_lims_df[~saffron_lims_df.date_of_acquisition.isin(saffron_co_df.raw_data_date)].copy()
saffron_to_upload_df = saffron_lims_df.iloc[6:-5].copy()
saffron_to_upload_df.shape

(12, 3)

In [58]:
saffron_to_upload_df

,ophys_session_id,date_of_acquisition,session_type
48,1347943754,2024-04-26,TRAINING_3_images_A_10uL_reward
56,1348625182,2024-04-29,TRAINING_3_images_A_10uL_reward
64,1363237428,2024-04-30,TRAINING_3_images_A_10uL_reward
104,1363633556,2024-05-02,TRAINING_4_images_A_training
80,1364402616,2024-05-06,TRAINING_5_images_A_epilogue
88,1364637859,2024-05-07,TRAINING_5_images_A_handoff_ready
96,1364904248,2024-05-08,OPHYS_1_images_A
113,1366153802,2024-05-13,OPHYS_1_images_A
121,1366398109,2024-05-14,OPHYS_4_images_B
137,1366915719,2024-05-16,OPHYS_4_images_B


In [59]:
lims_dirs = [str(from_lims.get_general_info_for_ophys_session_id(osid).session_storage_directory.values[0]) \
    for osid in saffron_to_upload_df.ophys_session_id.values]

In [60]:
saffron_upload_df = pd.DataFrame({'lims_directory': lims_dirs})
saffron_upload_df['capsule_id'] = '40e54443-c0cd-4403-adb7-d3e4c69b88de'
saffron_upload_df['mount'] = 'ophys_mount'
saffron_upload_df['force_cloud_sync'] = 'TRUE'

saffron_upload_df.to_csv(r"\\allen\programs\mindscope\workgroups\learning\saffron_to_upload_2025-02-13.csv", index=False)

In [9]:
# save_dir = Path(r'\\allen\programs\mindscope\workgroups\learning')
# saffron_df = exp_table.query('mouse_id == "721291"').reset_index()

# # there is one duplicate
# saffron_df = saffron_df.drop_duplicates(subset='ophys_experiment_id')
# saffron_df.to_csv(save_dir / '721291_experiments.csv', index=False)

# For LAMF data

In [19]:
exp_table.project_code.unique()

array(['VisualBehaviorMultiscope', 'LearningmFISHDevelopment', 'U01BFCT',
       'OpenScopeDendriteCoupling', 'VisualBehavior',
       'VisBIntTestDatacube', 'VisualBehaviorTask1B',
       'MultiscopeSignalNoise', 'omFISHRbp4Meso', 'LearningmFISHTask1A',
       'TaskTrainedNetworksMultiscope', 'omFISHGad2Meso',
       'VisualBehaviorMultiscope4areasx2d', 'omFISHSstMeso',
       'VipAxonalV1Phase1', 'MesoscopeDevelopment', 'omFISHCux2Meso',
       'VisualBehaviorIntegrationTest', 'VisualBehaviorDevelopment'],
      dtype=object)

In [25]:
lamf_table = exp_table.query('project_code == "LearningmFISHTask1A"')
np.sort(lamf_table.mouse_id.unique())

array(['603892', '608368', '612764', '616502', '616505', '617911',
       '624942', '629294', '633532', '636496', '639224', '646883',
       '648606', '662253', '662491', '671833', '677594', '681417',
       '681721', '690308', '693545', '700387', '701050', '703340',
       '704576', '710343', '711414'], dtype=object)

In [26]:
pre_sac_mouse_ids = ['603892', '608368', '612764', '616502', '616505', '617911',
       '624942', '629294', '633532', '636496', '639224', '646883']
post_sac_mouse_ids = np.setdiff1d(lamf_table.mouse_id.unique(), pre_sac_mouse_ids)
post_sac_mouse_ids

array(['648606', '662253', '662491', '671833', '677594', '681417',
       '681721', '690308', '693545', '700387', '701050', '703340',
       '704576', '710343', '711414'], dtype=object)

In [28]:
post_sac_table = lamf_table.query('mouse_id in @post_sac_mouse_ids')
post_sac_table

,equipment_name,donor_id,full_genotype,mouse_id,reporter_line,driver_line,sex,age_in_days,foraging_id,cre_line,...,session_name,isi_experiment_id,imaging_depth,targeted_structure,published_at,date_of_acquisition,session_type,experience_level,passive,image_set
ophys_experiment_id,,,,,,,,,,,,,,,,,,,,,
1275171146,MESO.2,1264107286,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,671833,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,99.0,6b989dd7-445d-4713-aa88-f323d44f5b49,Gad2-IRES-Cre,...,20230606_671833_training1,1269942361,71,VISp,NaT,2023-06-06 16:22:44.443,TRAINING_1_gratings,None,False,i
1275171148,MESO.2,1264107286,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,671833,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,99.0,6b989dd7-445d-4713-aa88-f323d44f5b49,Gad2-IRES-Cre,...,20230606_671833_training1,1269942361,371,VISp,NaT,2023-06-06 16:22:44.443,TRAINING_1_gratings,None,False,i
1275171151,MESO.2,1264107286,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,671833,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,99.0,6b989dd7-445d-4713-aa88-f323d44f5b49,Gad2-IRES-Cre,...,20230606_671833_training1,1269942361,271,VISam,NaT,2023-06-06 16:22:44.443,TRAINING_1_gratings,None,False,i
1275171152,MESO.2,1264107286,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,671833,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,99.0,6b989dd7-445d-4713-aa88-f323d44f5b49,Gad2-IRES-Cre,...,20230606_671833_training1,1269942361,71,VISam,NaT,2023-06-06 16:22:44.443,TRAINING_1_gratings,None,False,i
1275171145,MESO.2,1264107286,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,671833,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,99.0,6b989dd7-445d-4713-aa88-f323d44f5b49,Gad2-IRES-Cre,...,20230606_671833_training1,1269942361,265,VISp,NaT,2023-06-06 16:22:44.443,TRAINING_1_gratings,None,False,i
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259825184,MESO.1,1243805640,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,662253,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,106.0,24141fb1-2b21-4eec-a5ea-18fe5a20be1a,Gad2-IRES-Cre,...,20230403_662253_ophys6,1247177237,294,VISp,NaT,2023-04-03 16:27:41.544,OPHYS_6_images_B,Novel >1,False,B
1259825190,MESO.1,1243805640,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,662253,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,106.0,24141fb1-2b21-4eec-a5ea-18fe5a20be1a,Gad2-IRES-Cre,...,20230403_662253_ophys6,1247177237,262,VISam,NaT,2023-04-03 16:27:41.544,OPHYS_6_images_B,Novel >1,False,B
1259825187,MESO.1,1243805640,Gad2-IRES-Cre/wt;Slc32a1-T2A-FlpO/wt;Ai195(TIT...,662253,Ai195(TIT2L-GC7s-ICF-IRES-tTA2)-hyg,"[Slc32a1-T2A-FlpO, Gad2-IRES-Cre]",M,106.0,24141fb1-2b21-4eec-a5ea-18fe5a20be1a,Gad2-IRES-Cre,...,20230403_662253_ophys6,1247177237,379,VISp,NaT,2023-04-03 16:27:41.544,OPHYS_6_images_B,Novel >1,False,B


In [30]:
post_sac_table.columns

Index(['equipment_name', 'donor_id', 'full_genotype', 'mouse_id',
       'reporter_line', 'driver_line', 'sex', 'age_in_days', 'foraging_id',
       'cre_line', 'indicator', 'session_number',
       'prior_exposures_to_session_type', 'prior_exposures_to_image_set',
       'prior_exposures_to_omissions', 'ophys_session_id',
       'behavior_session_id', 'ophys_container_id', 'project_code',
       'container_workflow_state', 'experiment_workflow_state', 'session_name',
       'isi_experiment_id', 'imaging_depth', 'targeted_structure',
       'published_at', 'date_of_acquisition', 'session_type',
       'experience_level', 'passive', 'image_set'],
      dtype='object')

In [31]:
columns_to_use = ['ophys_session_id', 'mouse_id', 'equipment_name', 'date_of_acquisition', 'session_type', 'project_code', 'cre_line']
post_sac_table = post_sac_table[columns_to_use].set_index('ophys_session_id').drop_duplicates()
sorted_table = post_sac_table.sort_values(by=['mouse_id', 'date_of_acquisition'])
sorted_table['date'] = sorted_table.date_of_acquisition.apply(lambda x: pd.to_datetime(x).date())
sorted_table.drop('date_of_acquisition', axis=1, inplace=True)

## Filtering based on session progression
- At least one OPHYS_1 session

In [54]:
sorted_table.groupby('mouse_id').apply(lambda x: x.session_type.str.contains('OPHYS').sum())

mouse_id
648606    0
662253    6
662491    0
671833    0
677594    6
681417    5
681721    4
690308    0
693545    0
700387    3
701050    0
703340    6
704576    6
710343    0
711414    0
dtype: int64

In [59]:
tempseries = sorted_table.groupby('mouse_id').apply(lambda x: x.session_type.str.contains('OPHYS').sum())
use_mouse_ids = tempseries[tempseries > 0].index.values
print(len(use_mouse_ids))

7


In [60]:
upload_table = sorted_table.query('mouse_id in @use_mouse_ids')
print(len(upload_table))

115


In [58]:
save_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\pipeline_validation\code_ocean_uploading_temp')
save_dir.mkdir(exist_ok=True)
save_path = save_dir / 'post_sac_table_temp.csv'
upload_table.to_csv(save_path)

# For GCaMP pilot data
- mouse_id required (https://alleninstitute.sharepoint.com/:x:/s/LearningmFISH/EQn4cAgtbs5Fj18tV0oAwh0BtBecUuR_zFKNynz28B27FA?e=SxxUp3)
- To be updated here: https://alleninstitute.sharepoint.com/:x:/s/LearningmFISH/EaIR-C3-xaVGklhU9NumdBIBatt1tf0f9utLk4ZT_2gIgA?e=McEOD9

In [7]:
mouse_ids = [667825, 667826, 667827]
for mi, mid in enumerate(mouse_ids):
    if mi == 0:
        table = from_lims.get_imaging_ids_for_mouse_id(mid)
        table['mouse_id'] = [mid] * table.shape[0]
    else:
        assert isinstance(table, pd.DataFrame)
        new_table = from_lims.get_imaging_ids_for_mouse_id(mid)
        new_table['mouse_id'] = [mid] * new_table.shape[0]
        table = table.append(new_table)
table

C:\Users\jinho.kim\AppData\Local\Temp\ipykernel_6180\584621472.py:10: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  table = table.append(new_table)
C:\Users\jinho.kim\AppData\Local\Temp\ipykernel_6180\584621472.py:10: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  table = table.append(new_table)


,ophys_experiment_id,experiment_workflow_state,experiment_storage_directory,ophys_session_id,session_workflow_state,session_storage_directory,mouse_id
0,1265760434,passed,/allen/programs/mindscope/production/learning/...,1265619832,uploaded,/allen/programs/mindscope/production/learning/...,667825
1,1265760436,passed,/allen/programs/mindscope/production/learning/...,1265619832,uploaded,/allen/programs/mindscope/production/learning/...,667825
2,1263579241,passed,/allen/programs/mindscope/production/learning/...,1262862975,uploaded,/allen/programs/mindscope/production/learning/...,667825
3,1263579243,passed,/allen/programs/mindscope/production/learning/...,1262862975,uploaded,/allen/programs/mindscope/production/learning/...,667825
0,1267294087,processing,/allen/programs/mindscope/production/learning/...,1267183073,uploaded,/allen/programs/mindscope/production/learning/...,667826
1,1267294085,processing,/allen/programs/mindscope/production/learning/...,1267183073,uploaded,/allen/programs/mindscope/production/learning/...,667826
2,1263922955,processing,/allen/programs/mindscope/production/learning/...,1263821545,uploaded,/allen/programs/mindscope/production/learning/...,667826
3,1263922953,processing,/allen/programs/mindscope/production/learning/...,1263821545,uploaded,/allen/programs/mindscope/production/learning/...,667826
0,1265531034,failed,/allen/programs/mindscope/production/learning/...,1265369630,uploaded,/allen/programs/mindscope/production/learning/...,667827
1,1265531036,passed,/allen/programs/mindscope/production/learning/...,1265369630,uploaded,/allen/programs/mindscope/production/learning/...,667827


In [8]:
save_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\pilots\GCaMP8')
save_dir.mkdir(exist_ok=True)
save_path = save_dir / 'temp_table.csv'
table.to_csv(save_path, index=False)